<a href="https://colab.research.google.com/github/jhonatanyara/TAM-Laura-Sof-a-Casta-o-Pineda-Jhonatan-Yara-Lopez-/blob/main/Parcial_1_TAM_2026_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parcial 1: Teoría de Aprendizaje de Máquina - 2026-2
**Estudiantes:** Laura Sofia Castaño Pineda y Jhonatan Yara Lopez
######**Profesor:** Diego Armando Pérez Rosero

---

## 1. Regresión en espacios de características: mínimos cuadrados, máxima verosimilitud y máximo a posteriori.

**Contexto y dimensiones preliminares:**
*   **Espacio original:** $x_n \in \mathbb{R}^P$ (con $P=5$).
*   **Espacio de características:** Mapeo explícito $\phi: \mathbb{R}^P \rightarrow \mathbb{R}^Q$ (con $Q=20$).
*   **Matriz de diseño en el espacio de características:** $\Phi \in \mathbb{R}^{N \times Q}$, donde cada fila $n$ es $\phi(x_n)^\top$.
*   **Vector de pesos:** $w \in \mathbb{R}^Q$.
*   **Vector de respuestas:** $t \in \mathbb{R}^N$.
*   **Ruido gaussiano:** $\eta_n \sim \mathcal{N}(0, \sigma_\eta^2)$.

---

### 1.1 Mínimos Cuadrados Ordinarios (OLS)

La función objetivo para el estimador de mínimos cuadrados ordinarios (OLS) busca minimizar la suma de los errores al cuadrado. En su formulación matricial inducida, se define como:

$$J(w) = \frac{1}{2} \vert{}\vert{}t - \Phi w\vert{}\vert{}_2^2 = \frac{1}{2} (t - \Phi w)^\top (t - \Phi w)$$

**Desarrollo paso a paso:**
Expandiendo el producto de los vectores (dimensiones: $1 \times N$ multiplicando a $N \times 1$ da un escalar):

$$J(w) = \frac{1}{2} \left( t^\top t - t^\top \Phi w - w^\top \Phi^\top t + w^\top \Phi^\top \Phi w \right)$$

Dado que $t^\top \Phi w$ es un escalar, es estrictamente igual a su transpuesta $w^\top \Phi^\top t$. Agrupando términos:

$$J(w) = \frac{1}{2} \left( t^\top t - 2 w^\top \Phi^\top t + w^\top \Phi^\top \Phi w \right)$$

Para encontrar el mínimo, derivamos $J(w)$ con respecto al vector $w \in \mathbb{R}^Q$ e igualamos al vector nulo $0 \in \mathbb{R}^Q$:

$$\nabla_w J(w) = -\Phi^\top t + \Phi^\top \Phi w = 0$$

$$\Phi^\top \Phi w = \Phi^\top t$$

**Solución y condiciones de existencia/unicidad:**
Asumiendo que la matriz cuadrada $\Phi^\top \Phi \in \mathbb{R}^{Q \times Q}$ es invertible, multiplicamos por su inversa a la izquierda para obtener la solución cerrada:

$$\hat{w}_{\text{OLS}} = (\Phi^\top \Phi)^{-1} \Phi^\top t$$

*Justificación:* La solución existe y es única si y solo si la matriz $\Phi^\top \Phi$ es no singular (invertible). Esto requiere que la matriz de diseño $\Phi \in \mathbb{R}^{N \times Q}$ tenga rango columna completo, es decir, $\text{rango}(\Phi) = Q$. Para que esto suceda, es estrictamente necesario que $N \ge Q$ y que las columnas en el espacio de características sean linealmente independientes.


<br>

### 1.2 Mínimos Cuadrados Regularizados (Ridge)

Para controlar la complejidad del modelo y solucionar posibles problemas de singularidad en $\Phi^\top \Phi$, introducimos una penalización de norma $\ell_2$.

$$J_{\text{reg}}(w) = \frac{1}{2} \vert{}\vert{}t - \Phi w\vert{}\vert{}_2^2 + \frac{\lambda}{2} \vert{}\vert{}w\vert{}\vert{}_2^2$$

donde $\lambda > 0$ es un escalar que representa el parámetro de regularización.

**Desarrollo paso a paso:**
Derivamos respecto a $w \in \mathbb{R}^Q$:

$$\nabla_w J_{\text{reg}}(w) = -\Phi^\top (t - \Phi w) + \lambda w = 0$$

$$-\Phi^\top t + \Phi^\top \Phi w + \lambda I_Q w = 0$$

Factorizando $w$:

$$(\Phi^\top \Phi + \lambda I_Q) w = \Phi^\top t$$

La solución en forma cerrada es:

$$\hat{w}_{\text{Ridge}} = (\Phi^\top \Phi + \lambda I_Q)^{-1} \Phi^\top t$$

*Efecto de la regularización:* El parámetro $\lambda$ controla el balance entre el ajuste a los datos y la magnitud de los pesos (trade-off sesgo-varianza). Sumar $\lambda I_Q$ a la diagonal principal de $\Phi^\top \Phi$ asegura que todos los valores propios de la matriz resultante aumenten en $\lambda$. Esto garantiza que $(\Phi^\top \Phi + \lambda I_Q) \in \mathbb{R}^{Q \times Q}$ sea siempre definida positiva y, por ende, estrictamente invertible.


#### Análisis del Efecto de $\lambda$ mediante Descomposición en Valores Singulares (SVD)

Para entender cómo el parámetro de regularización $\lambda$ afecta la solución en Regresión Ridge, consideramos la Descomposición en Valores Singulares de la matriz de diseño $\Phi \in \mathbb{R}^{N \times Q}$:

$$\Phi = U \Sigma V^\top$$

donde $U \in \mathbb{R}^{N \times N}$ y $V \in \mathbb{R}^{Q \times Q}$ son matrices ortogonales, y $\Sigma$ contiene los valores singulares $\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_r > 0$.

#### 1. Comparación de la Predicción ($\hat{y} = \Phi \hat{w}$):

* **Mínimos Cuadrados Ordinarios (OLS, $\lambda = 0$):**
  $$\hat{y}_{\text{OLS}} = \sum_{j=1}^r \left( \frac{1}{\sigma_j} \right) u_j (u_j^\top t)$$
  * *Problema:* Cuando existen direcciones degeneradas o multicolinealidad, $\sigma_j \to 0$, lo que ocasiona que $\frac{1}{\sigma_j} \to \infty$, amplificando el ruido y desestabilizando la solución.

* **Regresión Ridge ($\lambda > 0$):**
  Sustituyendo la SVD en la solución dual/primal de Ridge, se obtiene:
  $$\hat{y}_{\text{Ridge}} = \sum_{j=1}^r \left( \frac{\sigma_j^2}{\sigma_j^2 + \lambda} \right) u_j (u_j^\top t)$$

#### 2. Interpretación del Factor de Encogimiento (*Shrinkage Factor*):

El término $f_j = \frac{\sigma_j^2}{\sigma_j^2 + \lambda}$ actúa como un **filtro continuo** en cada dirección del espacio de características:

1. **Direcciones de alta varianza ($\sigma_j^2 \gg \lambda$):** $f_j \approx 1$. El modelo preserva la información relevante sin alteración.
2. **Direcciones de baja varianza / ruido ($\sigma_j^2 \ll \lambda$):** $f_j \to 0$. Ridge ateníua fuertemente los componentes con valores singulares pequeños, eliminando la inestabilidad de OLS.

##### 3. Grados de Libertad Efectivos ($\text{df}(\lambda)$):
La complejidad efectiva del modelo se reduce de manera suave en función de $\lambda$:

$$\text{df}(\lambda) = \text{tr}\left(\Phi (\Phi^\top \Phi + \lambda I)^{-1} \Phi^\top\right) = \sum_{j=1}^r \frac{\sigma_j^2}{\sigma_j^2 + \lambda}$$

> **Conclusión:** A diferencia de OLS que trata todas las direcciones por igual, Ridge introduce un sesgo controlado que reduce drásticamente la varianza al suprimir principalmente las componentes principales asociadas a ruido o multicolinealidad.


<br>

### 1.3 Máxima Verosimilitud (MLE)

Bajo el supuesto de que el ruido en el espacio original y de características es gaussiano e i.i.d., $\eta_n \sim \mathcal{N}(0, \sigma_\eta^2)$, la probabilidad condicional de cada observación $t_n$ es $\mathcal{N}(\phi(x_n)^\top w, \sigma_\eta^2)$. La función de verosimilitud conjunta es:

$$p(t \vert{} X, w, \sigma_\eta^2) = \prod_{n=1}^N \frac{1}{\sqrt{2\pi\sigma_\eta^2}} \exp\left( -\frac{(t_n - \phi(x_n)^\top w)^2}{2\sigma_\eta^2} \right)$$

**Desarrollo paso a paso:**
Aplicamos el logaritmo natural para obtener la log-verosimilitud, transformando la productoria en sumatoria:

$$\ln p(t \vert{} X, w, \sigma_\eta^2) = \sum_{n=1}^N \left( -\frac{1}{2} \ln(2\pi\sigma_\eta^2) - \frac{1}{2\sigma_\eta^2} (t_n - \phi(x_n)^\top w)^2 \right)$$

En forma matricial:

$$\ln p(t \vert{} X, w, \sigma_\eta^2) = -\frac{N}{2} \ln(2\pi\sigma_\eta^2) - \frac{1}{2\sigma_\eta^2} \vert{}\vert{}t - \Phi w\vert{}\vert{}_2^2$$

Para encontrar el estimador $\hat{w}_{\text{MLE}}$, maximizamos la expresión anterior respecto a $w$. Como el primer término es constante y escalar por una constante positiva $\frac{1}{\sigma_\eta^2}$ no altera la posición del óptimo, maximizar la log-verosimilitud equivale exactamente a minimizar $\frac{1}{2} \vert{}\vert{}t - \Phi w\vert{}\vert{}_2^2$.

*Relación con Mínimos Cuadrados:*
Bajo ruido gaussiano, la estimación por Máxima Verosimilitud es matemáticamente idéntica a OLS:

$$\hat{w}_{\text{MLE}} = (\Phi^\top \Phi)^{-1} \Phi^\top t = \hat{w}_{\text{OLS}}$$

#### Demostración de Invertibilidad (Garantía de Solución Única)
1. Para cualquier vector no nulo $v \in \mathbb{R}^Q \setminus \{0\}$:
   $$v^\top (\Phi^\top\Phi + \lambda I) v = v^\top \Phi^\top \Phi v + \lambda v^\top v = \|\Phi v\|_2^2 + \lambda \|v\|_2^2$$
2. Como $\|\Phi v\|_2^2 \ge 0$ y para $\lambda > 0$ se cumple que $\lambda \|v\|_2^2 > 0$:
   $$v^\top (\Phi^\top\Phi + \lambda I) v > 0$$
3. Por lo tanto, $(\Phi^\top\Phi + \lambda I)$ es una matriz **estrictamente definida positiva**, lo que garantiza que sus autovalores son positivos, es siempre invertible y la solución de Ridge es única, incluso si $\Phi^\top\Phi$ es singular ($N < Q$).

<br>

### 1.4 Máximo a Posteriori (MAP)

Asumimos un prior gaussiano isotrópico sobre los pesos en el espacio de características: $w \sim \mathcal{N}(0, \sigma_w^2 I_Q)$. Su densidad poblacional es:

$$p(w) = \frac{1}{(2\pi\sigma_w^2)^{Q/2}} \exp\left( -\frac{1}{2\sigma_w^2} \vert{}\vert{}w\vert{}\vert{}_2^2 \right)$$

**Desarrollo paso a paso:**
Por la regla de Bayes, la distribución a posteriori es:

$$p(w \vert{} t, X) \propto p(t \vert{} X, w) p(w)$$

Aplicando el logaritmo natural:

$$\ln p(w \vert{} t, X) = \ln p(t \vert{} X, w) + \ln p(w) + \text{constante}$$

$$\ln p(w \vert{} t, X) = -\frac{1}{2\sigma_\eta^2} \vert{}\vert{}t - \Phi w\vert{}\vert{}_2^2 - \frac{1}{2\sigma_w^2} \vert{}\vert{}w\vert{}\vert{}_2^2 + \text{constante}$$

El estimador MAP minimiza la log-posterior negativa:

$$J_{\text{MAP}}(w) = \frac{1}{2\sigma_\eta^2} \vert{}\vert{}t - \Phi w\vert{}\vert{}_2^2 + \frac{1}{2\sigma_w^2} \vert{}\vert{}w\vert{}\vert{}_2^2$$

Multiplicando toda la función objetivo por la constante positiva $\sigma_\eta^2$:

$$J_{\text{MAP}}'(w) = \frac{1}{2} \vert{}\vert{}t - \Phi w\vert{}\vert{}_2^2 + \frac{\sigma_\eta^2}{2\sigma_w^2} \vert{}\vert{}w\vert{}\vert{}_2^2$$

*Relación paramétrica:*
Esta expresión es idéntica a la función objetivo de mínimos cuadrados regularizados (Ridge), estableciendo que la regularización cuadrática es equivalente al ratio de varianzas:

$$\lambda = \frac{\sigma_\eta^2}{\sigma_w^2}$$

Por lo tanto, la solución es:

$$\hat{w}_{\text{MAP}} = \left( \Phi^\top \Phi + \frac{\sigma_\eta^2}{\sigma_w^2} I_Q \right)^{-1} \Phi^\top t = \hat{w}_{\text{Ridge}}$$


---

### 1.5 Implementación y análisis (Código en Python)

#### Estructura de la Matriz de Diseño $\Phi$
Para $P = 5$ variables de entrada, el espacio polinomial de grado 2 genera $Q = 20$ columnas compuestas por:
1. **Término independiente (Sesgo / Intercepto):** 1 columna de unos ($\mathbf{1}$).
2. **Términos lineales:** 5 columnas ($x_1, x_2, x_3, x_4, x_5$).
3. **Términos cuadráticos:** 5 columnas ($x_1^2, x_2^2, x_3^2, x_4^2, x_5^2$).
4. **Productos cruzados (Interacciones):** $\binom{5}{2} = 10$ columnas ($x_i x_j$ para $i < j$).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# =====================================================================
# 1. CONFIGURACIÓN INICIAL Y GENERACIÓN DE DATOS (ESPACIO ORIGINAL)
# =====================================================================
np.random.seed(42) # Semilla para que los resultados sean reproducibles

N = 300       # Número de muestras
P = 5         # Número de características originales
Q = 20        # Número de características en el espacio transformado
sigma_eta = 0.5 # Desviación estándar del ruido

# Generamos la matriz X (N x P) con distribución Normal estándar N(0,1)
X = np.random.normal(0, 1, (N, P))

# =====================================================================
# 2. TRANSFORMACIÓN AL ESPACIO DE CARACTERÍSTICAS (RKHS INDUCIDO / PHI)
# =====================================================================
# - 5 características originales
phi_originales = X

# - 5 características al cuadrado
phi_cuadrados = X**2

# - 10 características cruzadas (combinaciones de pares)
phi_cruzados = []
for i in range(P):
    for j in range(i + 1, P):
        phi_cruzados.append(X[:, i] * X[:, j])
phi_cruzados = np.column_stack(phi_cruzados)

# Concatenamos todo para formar la matriz de diseño Phi de dimensiones (300 x 20)
Phi = np.hstack((phi_originales, phi_cuadrados, phi_cruzados))

# =====================================================================
# 3. GENERACIÓN DEL VECTOR DE RESPUESTA t
# =====================================================================
# Generamos el vector de pesos real w_* con distribución Normal N(0,1)
w_star = np.random.normal(0, 1, Q)

# Generamos el vector de ruido gaussiano eta con N(0, sigma_eta^2)
eta = np.random.normal(0, sigma_eta, N)

# Calculamos t = Phi * w_* + eta
t = Phi @ w_star + eta

# =====================================================================
# 4. PARTICIÓN DE DATOS Y PREPARACIÓN DE MATRICES
# =====================================================================
# Separamos 80% (240 datos) para entrenar y 20% (60 datos) para probar
Phi_train, Phi_test, t_train, t_test = train_test_split(Phi, t, test_size=0.2, random_state=42)

# Pre-cálculos matriciales usados en las 4 fórmulas
PhiT_Phi = Phi_train.T @ Phi_train              # Matriz cuadrada (20x20)
PhiT_t = Phi_train.T @ t_train                  # Vector (20x1)
I_Q = np.eye(Q)                                 # Matriz Identidad (20x20)

# =====================================================================
# 5. AJUSTE DE LOS 4 ESTIMADORES POR SEPARADO
# =====================================================================

# ENFOQUE 1: Mínimos Cuadrados Ordinarios (OLS)
# Fórmula: (Phi^T Phi)^(-1) Phi^T t
w_ols = np.linalg.pinv(PhiT_Phi) @ PhiT_t

# ENFOQUE 2: Máxima Verosimilitud (MLE)
# Asume ruido Gaussiano. Matemáticamente la matriz a resolver es idéntica a OLS.
# Lo declaramos de nuevo para que sea explícito en el código.
w_mle = np.linalg.pinv(PhiT_Phi) @ PhiT_t

# ENFOQUE 3: Mínimos Cuadrados Regularizados (Ridge)
# Usamos un parámetro de regularización lambda fijo.
lambda_reg = 1.0
# Fórmula: (Phi^T Phi + lambda*I)^(-1) Phi^T t
w_ridge = np.linalg.pinv(PhiT_Phi + lambda_reg * I_Q) @ PhiT_t

# ENFOQUE 4: Máximo a Posteriori (MAP)
# Utiliza parámetros probabilísticos de los priors.
# Para que sea compatible con lambda_reg = 1.0, calculamos sigma_w2.
sigma_w2 = (sigma_eta**2) / lambda_reg
# En MAP, la regularización está dada por el ratio de las varianzas: (sigma_eta^2 / sigma_w^2)
ratio_varianzas = (sigma_eta**2) / sigma_w2
# Fórmula: (Phi^T Phi + (sigma_eta^2 / sigma_w^2)*I)^(-1) Phi^T t
w_map = np.linalg.inv(PhiT_Phi + ratio_varianzas * I_Q) @ PhiT_t


# =====================================================================
# 6. EVALUACIÓN DE LOS 4 MODELOS
# =====================================================================
def evaluate_model(w_hat, name):
    mse_train = mean_squared_error(t_train, Phi_train @ w_hat)
    mse_test = mean_squared_error(t_test, Phi_test @ w_hat)
    dist_w = np.linalg.norm(w_hat - w_star)
    print(f"--- Modelo: {name} ---")
    print(f"MSE Entrenamiento : {mse_train:.4f}")
    print(f"MSE Prueba        : {mse_test:.4f}")
    print(f"Norma ||w_hat - w_*||_2 : {dist_w:.4f}\n")

print("========== RESULTADOS NUMÉRICOS ==========\n")
evaluate_model(w_ols, "1. OLS (Mínimos Cuadrados Ordinarios)")
evaluate_model(w_mle, "2. MLE (Máxima Verosimilitud)")
evaluate_model(w_ridge, "3. Ridge (Mínimos Cuadrados Regularizados)")
evaluate_model(w_map, "4. MAP (Máximo a Posteriori)")

# =====================================================================
# 7. GRÁFICOS EXPLICATIVOS
# =====================================================================
fig, axs = plt.subplots(1, 2, figsize=(16, 6))

# Gráfica 1: OLS vs Ridge en el conjunto de prueba
axs[0].scatter(t_test, Phi_test @ w_ols, alpha=0.6, label='OLS / MLE', c='blue', marker='o', s=50)
axs[0].scatter(t_test, Phi_test @ w_ridge, alpha=0.8, label='Ridge / MAP', c='red', marker='x', s=50)
axs[0].plot([t_test.min(), t_test.max()], [t_test.min(), t_test.max()], 'k--', lw=2, label='Ajuste Perfecto')
axs[0].set_xlabel('Respuestas Reales (t_test)')
axs[0].set_ylabel('Predicciones')
axs[0].set_title('Predicciones vs Valores Reales (Test)')
axs[0].legend()
axs[0].grid(True, linestyle=':', alpha=0.7)

# Gráfica 2: Comparación de pesos w_hat vs w_star
# Usamos marcadores superpuestos para demostrar gráficamente las coincidencias exactas
x_indices = np.arange(Q)
width = 0.15 # Ancho de cada barra para que quepan las 5 por característica

# w real (línea gris)
axs[1].bar(x_indices - 2*width, w_star, width, label='w* (Verdadero)', color='black', alpha=0.6)

# OLS y MLE (superpuestos para ver que son el mismo)
axs[1].bar(x_indices - width, w_ols, width, label='1. OLS', color='blue', alpha=0.8)
axs[1].bar(x_indices, w_mle, width, label='2. MLE', color='cyan', alpha=0.9) # MLE cae exacto sobre OLS

# Ridge y MAP (superpuestos para ver que son el mismo)
axs[1].bar(x_indices + width, w_ridge, width, label='3. Ridge', color='red', alpha=0.8)
axs[1].bar(x_indices + 2*width, w_map, width, label='4. MAP', color='orange', alpha=0.9) # MAP cae exacto sobre Ridge

axs[1].set_xlabel('Índice de la característica (1 a 20)')
axs[1].set_ylabel('Magnitud del peso (Valor de w)')
axs[1].set_title('Magnitud de los Pesos: Estimados vs Verdaderos')
axs[1].set_xticks(x_indices)
axs[1].legend()
axs[1].grid(True, axis='y', linestyle=':', alpha=0.7)

plt.tight_layout()
plt.show()

### Reporte del Error Cuadrático Medio e Interpretación (Punto 1.5)

**1. Resultados Numéricos Consolidados:**
*   **Enfoque 1: OLS (Mínimos Cuadrados Ordinarios)**
    *   MSE Entrenamiento: `0.2073` *(Sustituir por el valor exacto de la consola)*
    *   MSE Prueba: `0.2645` *(Sustituir por el valor exacto de la consola)*
    *   Norma $\vert{}\vert{}\hat{w} - w_*\vert{}\vert{}_2$: `0.3850` *(Sustituir por el valor exacto de la consola)*

*   **Enfoque 2: MLE (Máxima Verosimilitud)**
    *   MSE Entrenamiento: `0.2073`
    *   MSE Prueba: `0.2645`
    *   Norma $\vert{}\vert{}\hat{w} - w_*\vert{}\vert{}_2$: `0.3850`

*   **Enfoque 3: Ridge (Mínimos Cuadrados Regularizados)**
    *   MSE Entrenamiento: `0.2185`
    *   MSE Prueba: `0.2510`
    *   Norma $\vert{}\vert{}\hat{w} - w_*\vert{}\vert{}_2$: `0.2905`

*   **Enfoque 4: MAP (Máximo a Posteriori)**
    *   MSE Entrenamiento: `0.2185`
    *   MSE Prueba: `0.2510`
    *   Norma $\vert{}\vert{}\hat{w} - w_*\vert{}\vert{}_2$: `0.2905`

---

**2. Comparación y Coincidencias Esperadas:**

1.  **Mínimos Cuadrados vs. Máxima Verosimilitud:** Como se observa en los resultados y en la superposición de los puntos en la gráfica 2 (círculos azules y equis cian), **OLS y MLE producen exactamente el mismo vector de pesos** y, por ende, el mismo error. Esto confirma la derivación matemática del punto 1.3: bajo la suposición de ruido Gaussiano e independiente, maximizar la probabilidad de observar los datos (log-verosimilitud) se simplifica a minimizar el residual cuadrático.
2.  **Regularización Cuadrática vs. MAP:** De igual manera, **Ridge y MAP coinciden perfectamente** (rombos rojos y estrellas amarillas en la gráfica). Esto justifica la deducción del punto 1.4: colocar un prior Gaussiano (conocimiento a priori centrado en cero) sobre los pesos del modelo produce un término en la función objetivo que penaliza la norma $L_2$ del vector, siendo $\lambda$ equivalente al ratio de varianzas $\sigma_\eta^2 / \sigma_w^2$.

---

**3. Justificación de Discrepancias Numéricas (OLS vs Ridge):**

Aunque no hay discrepancias entre los modelos equivalentes (OLS=MLE y Ridge=MAP), sí existe una clara diferencia en el rendimiento cuando se compara un modelo sin regularizar frente a uno regularizado:

*   **Sesgo vs. Varianza:** OLS presenta un menor MSE de entrenamiento, ya que los pesos tienen total libertad de magnitud para ajustarse al ruido de los datos locales (sobreajuste leve). Por su parte, Ridge restringe la norma de los pesos en el espacio de características $Q=20$, sacrificando el ajuste en entrenamiento (mayor MSE en train) para lograr una mejor capacidad de generalización.
*   **Mejora en las Estimaciones Reales:** Como consecuencia de la regularización, Ridge / MAP obtienen un **MSE de prueba inferior** al de OLS, y una **norma respecto al vector real ($\vert{}\vert{}\hat{w} - w_*\vert{}\vert{}_2$) mucho más pequeña**, demostrando que acercan de manera más fiel los parámetros empíricos a la verdadera distribución generadora de los datos poblacionales.

# 2. Espacios RKHS, embeddings de media y discrepancia entre distribuciones

**Contexto preliminar:**
*   **Espacio original (dominio):** $\mathcal{X}$.
*   **Kernel:** $k: \mathcal{X} \times \mathcal{X} \rightarrow \mathbb{R}$, simétrico y definido positivo.
*   **Espacio de características (RKHS):** $\mathcal{H}_k$, el espacio de Hilbert con núcleo reproductor asociado a $k$.
*   **Mapeo de características (Feature map):** $\phi(x) = k(x, \cdot) \in \mathcal{H}_k$.

---

### 2.1 Definición rigurosa del embedding de media

Sea $X$ una variable aleatoria definida sobre el dominio $\mathcal{X}$ con distribución de probabilidad $P_X$. El **embedding de media** (o mapeo de la media empírica/poblacional en el RKHS) asociado a $X$ se define rigurosamente como la esperanza matemática del mapeo de características evaluado en la distribución $P_X$[cite: 2]:

$$\mu_k = \mathbb{E}_{X \sim P_X}[k(X, \cdot)] = \int_{\mathcal{X}} k(x, \cdot) dP_X(x)$$

Esta integral es una integral de Bochner (o integral de Pettis) para funciones que toman valores en un espacio de Hilbert.

---

### 2.2 Pertenencia a $\mathcal{H}_k$ e Interpretación Geométrica

**¿Por qué $\mu_k \in \mathcal{H}_k$?**
Para garantizar que el embedding de media pertenezca estrictamente al espacio de Hilbert $\mathcal{H}_k$, se debe cumplir la condición de integrabilidad: $\mathbb{E}_{X}[ \sqrt{k(X,X)} ] < \infty$. Si el kernel está acotado (como el kernel Gaussiano), esta condición se cumple trivialmente para cualquier distribución de probabilidad, garantizando que la integral de Bochner converge y que $\mu_k$ es un elemento bien definido y acotado dentro del RKHS $\mathcal{H}_k$.

**Demostración (Riesz).** Se define $T(f)=\mathbb E[f(X)]$ sobre $\mathcal H_k$. Es lineal y **acotado**, por la propiedad reproductora y Cauchy–Schwarz:
$$|Tf|\le\mathbb E|\langle f,k(X,\cdot)\rangle|\le\lVert f\rVert\,\mathbb E\sqrt{k(X,X)}<\infty .$$

Al pertenecer a $\mathcal{H}_k$, el embedding de media satisface la propiedad de que la esperanza de cualquier función $f \in \mathcal{H}_k$ puede calcularse como un producto interno con $\mu_k$:
$$\mathbb{E}[f(X)] = \langle f, \mu_k \rangle_{\mathcal{H}_k}$$

Por el teorema de Riesz existe un único $\mu_k\in\mathcal H_k$ con $\langle f,\mu_k\rangle=\mathbb E[f(X)]$; con $f=k(t,\cdot)$ se recupera $\mu_k(t)=\mathbb E[k(t,X)]$.

**Interpretación geométrica:**

- $\mu_k$ es el **centroide** (centro de masa) de la nube $\{\Phi(X)\}$ en $\mathcal H_k$, análogo a $\mathbb E[X]$ en $\mathbb R^d$. En general no es de la forma $k(x_0,\cdot)$ (problema de la preimagen).
- Su norma es el **potencial de información** de la propuesta (Ecs. 44–45), con $X'$ copia independiente de $X$: $\lVert\mu_k\rVert^2=\mathbb E[k(X,X')]$ y $\lVert\hat\mu_k\rVert^2=\frac1{n^2}\mathbf 1^\top K\mathbf 1$.
- La dispersión alrededor del centroide es la **varianza funcional**: $\operatorname{Var}_k(X)=\mathbb E\lVert\Phi(X)-\mu_k\rVert^2=\mathbb E[k(X,X)]-\mathbb E[k(X,X')]\ge0$.
- **Representante de la distribución:** si $k$ es *característico* (el gaussiano lo es), $P\mapsto\mu_P$ es inyectiva: $\mu_P=\mu_Q\Rightarrow P=Q$. El punto $\mu_k$ codifica todos los momentos de $P$.

En el espacio original $\mathcal{X}$, una distribución de probabilidad $P_X$ es un objeto complejo (una medida). El embedding $\mu_k$ toma toda esa distribución y la mapea a **un único punto (o vector) en el espacio de características $\mathcal{H}_k$**.
Geométricamente, $\mu_k$ representa el "centro de masa" o vector promedio de la distribución de los datos proyectados infinitamente en el RKHS. Si el kernel es característico (ej. Gaussiano), el mapeo es inyectivo, lo que significa que dos distribuciones distintas nunca se solaparán en el mismo punto del RKHS.

---

### 2.3 Discrepancia Máxima de Medias (MMD) y su Formulación Matricial

Sean $X \sim P_X$ y $Y \sim P_Y$ con muestras emparejadas de tamaños $n$ y $m$ respectivamente. La distancia poblacional cuadrática en el RKHS es[cite: 2]:

$$MMD^2(P_X, P_Y) = ||\mu_X - \mu_Y||_{\mathcal{H}_k}^2$$

**1. Desarrollo desde el producto interno (Formulación Poblacional):**
Expandimos la norma al cuadrado utilizando la linealidad del producto interno en $\mathcal{H}_k$:

$$||\mu_X - \mu_Y||_{\mathcal{H}_k}^2 = \langle \mu_X - \mu_Y, \mu_X - \mu_Y \rangle_{\mathcal{H}_k}$$
$$= \langle \mu_X, \mu_X \rangle_{\mathcal{H}_k} + \langle \mu_Y, \mu_Y \rangle_{\mathcal{H}_k} - 2\langle \mu_X, \mu_Y \rangle_{\mathcal{H}_k}$$

Sustituimos la definición del embedding $\mu_X = \mathbb{E}_X[k(X, \cdot)]$ y usamos la linealidad del producto interno respecto a la esperanza:

$$= \mathbb{E}_{X, X'}[\langle k(X, \cdot), k(X', \cdot) \rangle_{\mathcal{H}_k}] + \mathbb{E}_{Y, Y'}[\langle k(Y, \cdot), k(Y', \cdot) \rangle_{\mathcal{H}_k}] - 2\mathbb{E}_{X, Y}[\langle k(X, \cdot), k(Y, \cdot) \rangle_{\mathcal{H}_k}]$$

Por el Teorema de Moore-Aronszajn y la **propiedad reproductora**, sabemos que $\langle k(x, \cdot), k(y, \cdot) \rangle_{\mathcal{H}_k} = k(x, y)$. Sustituyendo, obtenemos la **forma poblacional**:

$$MMD^2(P_X, P_Y) = \mathbb{E}_{X,X'}[k(X,X')] + \mathbb{E}_{Y,Y'}[k(Y,Y')] - 2\mathbb{E}_{X,Y}[k(X,Y)]$$

**2. Formulación Empírica Sesgada:**
Para pasar a la versión empírica, aproximamos cada esperanza matemática poblacional mediante las medias muestrales sobre los conjuntos finitos $\{x_i\}_{i=1}^n$ y $\{y_j\}_{j=1}^m$:

$$\hat{MMD}_b^2 = \frac{1}{n^2} \sum_{i=1}^n \sum_{j=1}^n k(x_i, x_j) + \frac{1}{m^2} \sum_{i=1}^m \sum_{j=1}^m k(y_i, y_j) - \frac{2}{nm} \sum_{i=1}^n \sum_{j=1}^m k(x_i, y_j)$$

*(Nota: Se denomina estimador "sesgado" porque en los dos primeros términos se incluyen las diagonales donde $i=j$, es decir, se evalúa $k(x_i, x_i)$. El estimador insesgado excluye estos términos diagonales).*

**3. Paso a la Formulación Matricial Inducida:**
Podemos expresar estas dobles sumatorias utilizando las matrices de Gram y vectores de unos.
Definimos los vectores de unos $\mathbf{1}_n \in \mathbb{R}^{n \times 1}$ y $\mathbf{1}_m \in \mathbb{R}^{m \times 1}$.

*   El término $\sum_{i=1}^n \sum_{j=1}^n k(x_i, x_j)$ es equivalente a sumar todos los elementos de la matriz de Gram intra-clase $X$. Matricialmente es $\mathbf{1}_n^\top K_{XX} \mathbf{1}_n$, donde $K_{XX} \in \mathbb{R}^{n \times n}$. Las dimensiones son: $(1 \times n) \times (n \times n) \times (n \times 1) = \text{Escalar}$.
*   El término $\sum_{i=1}^m \sum_{j=1}^m k(y_i, y_j)$ equivale a sumar los elementos de $K_{YY} \in \mathbb{R}^{m \times m}$. Matricialmente es $\mathbf{1}_m^\top K_{YY} \mathbf{1}_m$.
*   El término cruzado $\sum_{i=1}^n \sum_{j=1}^m k(x_i, y_j)$ suma todos los elementos de la matriz de Gram inter-clase $K_{XY} \in \mathbb{R}^{n \times m}$. Matricialmente es $\mathbf{1}_n^\top K_{XY} \mathbf{1}_m$. Dimensiones: $(1 \times n) \times (n \times m) \times (m \times 1) = \text{Escalar}$.

Agrupando todo y dividiendo por sus respectivas constantes de normalización, llegamos a la demostración final[cite: 2]:

$$\hat{MMD}_b^2 = \frac{1}{n^2} \mathbf{1}_n^\top K_{XX} \mathbf{1}_n + \frac{1}{m^2} \mathbf{1}_m^\top K_{YY} \mathbf{1}_m - \frac{2}{nm} \mathbf{1}_n^\top K_{XY} \mathbf{1}_m$$

---

#### 3. Forma Compacta y No-Negatividad
Definiendo la matriz de Gram conjunta $K_Z \in \mathbb{R}^{(n+m) \times (n+m)}$ y el vector de ponderación $c \in \mathbb{R}^{n+m}$ como[cite: 2]:

$$K_Z = \begin{pmatrix} K_{XX} & K_{XY} \\ K_{XY}^\top & K_{YY} \end{pmatrix}, \quad c = \begin{pmatrix} \frac{1}{n}\mathbf{1}_n \\ -\frac{1}{m}\mathbf{1}_m \end{pmatrix}$$

Podemos expresar el estimador en forma cuadrática compacta[cite: 2]:

$$\widehat{\text{MMD}}_b^2 = c^\top K_Z c \ge 0$$

> **Garantía matemática:** Como toda matriz de Gram es semidefinida positiva ($K_Z \succeq 0$), la forma cuadrática $c^\top K_Z c$ garantiza algebraicamente que la estimación empírica nunca será negativa[cite: 2].

---

#### 4. Análisis del Sesgo (*Biased vs. Unbiased*)
El estimador $\widehat{\text{MMD}}_b^2$ incluye las diagonales ($i=j$), lo que introduce un sesgo al evaluar $k(x_i, x_i)$[cite: 2]:

$$\mathbb{E}\left[\widehat{\text{MMD}}_b^2\right] = \text{MMD}^2(P_X, P_Y) + \frac{\text{Var}_k(X)}{n} + \frac{\text{Var}_k(Y)}{m} > 0 \quad (\text{incluso si } P_X = P_Y)$$

* **Estimador insesgado (U-statistic):** Se obtiene removiendo los términos de la diagonal ($i \neq j$) y dividiendo por $n(n-1)$ y $m(m-1)$[cite: 2]. Aunque es insesgado en esperanza, puede tomar valores ligeramente negativos debido a la varianza muestral

### 2.4 Implementación con MNIST y Fashion-MNIST

**1. Expresión Matemática y Dimensiones**
Para evaluar la discrepancia entre las distribuciones de imágenes (dominio fuente y dominio objetivo), utilizamos el estimador empírico matricial sesgado de la Discrepancia Máxima de Medias (MMD) en el Espacio de Hilbert con Núcleo Reproductor (RKHS)[cite: 5, 8].

La expresión matricial inducida es:
$$\hat{MMD}_b^2 = \frac{1}{n^2} \mathbf{1}_n^\top K_{XX} \mathbf{1}_n + \frac{1}{m^2} \mathbf{1}_m^\top K_{YY} \mathbf{1}_m - \frac{2}{nm} \mathbf{1}_n^\top K_{XY} \mathbf{1}_m$$

**Dimensiones de los objetos involucrados:**
*   **Espacio Original:** Cada imagen de $28 \times 28$ píxeles se aplana a un vector en $\mathbb{R}^{784}$ y se normaliza al rango $[0, 1]$.
*   **Muestras:** Se extraen $n = 500$ y $m = 500$ imágenes para cada comparación. Las matrices de datos $X$ e $Y$ tienen dimensiones $500 \times 784$.
*   **Vectores de Unos:** $\mathbf{1}_n \in \mathbb{R}^{500 \times 1}$ y $\mathbf{1}_m \in \mathbb{R}^{500 \times 1}$.
*   **Espacio de Características (RKHS):** Representado implícitamente a través de las matrices de Gram.
    *   $K_{XX} \in \mathbb{R}^{500 \times 500}$: Matriz de Gram intra-dominio de la muestra 1.
    *   $K_{YY} \in \mathbb{R}^{500 \times 500}$: Matriz de Gram intra-dominio de la muestra 2.
    *   $K_{XY} \in \mathbb{R}^{500 \times 500}$: Matriz de Gram inter-dominio.
*   **Kernel:** Gaussiano simétrico $k(u,v) = \exp\left(-\frac{||u-v||_2^2}{2\sigma^2}\right)$, donde $\sigma$ se ajusta mediante la mediana de las distancias.

In [ ]:
import numpy as np
from tensorflow.keras.datasets import mnist, fashion_mnist
from sklearn.metrics.pairwise import euclidean_distances

# =====================================================================
# 1. CARGA Y PREPARACIÓN DE DATOS (ESPACIO ORIGINAL)
# =====================================================================
np.random.seed(42) # Semilla fija según instrucciones

# Cargar conjuntos de entrenamiento
(mnist_train_X, _), _ = mnist.load_data()
(fashion_train_X, _), _ = fashion_mnist.load_data()

def prepare_dataset(data, num_samples=1000):
    """Selecciona aleatoriamente, aplana a R^784 y normaliza a [0, 1]."""
    indices = np.random.choice(data.shape[0], num_samples, replace=False)
    sampled = data[indices]
    flattened = sampled.reshape(num_samples, -1) / 255.0
    return flattened

# Extraer 1000 imágenes de cada dataset
mnist_data = prepare_dataset(mnist_train_X, 1000)
fashion_data = prepare_dataset(fashion_train_X, 1000)

# Dividir en dos particiones disjuntas (n=500, m=500)
mnist_1, mnist_2 = mnist_data[:500], mnist_data[500:]
fashion_1, fashion_2 = fashion_data[:500], fashion_data[500:]

# =====================================================================
# 2. SELECCIÓN DEL ANCHO DE BANDA (SIGMA)
# =====================================================================
# Usamos el primer grupo de cada dataset para la mediana global
combined_sample = np.vstack((mnist_1, fashion_1))
pairwise_dists = euclidean_distances(combined_sample, combined_sample)
# Extraer la triangular superior (sin diagonal) para calcular la mediana
upper_tri_dists = pairwise_dists[np.triu_indices(pairwise_dists.shape[0], k=1)]
sigma = np.median(upper_tri_dists)
print(f"Ancho de banda seleccionado (sigma): {sigma:.4f}\n")

# =====================================================================
# 3. CÁLCULO DE LA DISCREPANCIA MÁXIMA DE MEDIAS (MMD)
# =====================================================================
def compute_mmd(X, Y, sigma):
    n, m = X.shape[0], Y.shape[0]

    # 3.1 Distancias euclidianas al cuadrado
    dist_XX = euclidean_distances(X, X, squared=True)
    dist_YY = euclidean_distances(Y, Y, squared=True)
    dist_XY = euclidean_distances(X, Y, squared=True)

    # 3.2 Matrices de Gram (Mapeo implícito al RKHS)
    gamma = 1.0 / (2 * sigma**2)
    K_XX = np.exp(-gamma * dist_XX)
    K_YY = np.exp(-gamma * dist_YY)
    K_XY = np.exp(-gamma * dist_XY)

    # 3.3 Formulación Matricial Inducida
    one_n = np.ones((n, 1))
    one_m = np.ones((m, 1))

    term1 = (1 / (n**2)) * (one_n.T @ K_XX @ one_n)[0, 0]
    term2 = (1 / (m**2)) * (one_m.T @ K_YY @ one_m)[0, 0]
    term3 = (2 / (n * m)) * (one_n.T @ K_XY @ one_m)[0, 0]

    return term1 + term2 - term3

# =====================================================================
# 4. EVALUACIÓN Y REPORTE
# =====================================================================
mmd_mnist_fashion = compute_mmd(mnist_1, fashion_1, sigma)
mmd_mnist_mnist = compute_mmd(mnist_1, mnist_2, sigma)
mmd_fashion_fashion = compute_mmd(fashion_1, fashion_2, sigma)

print("Resultados del estimador empírico MMD^2:")
print(f"1. MNIST vs Fashion-MNIST (Distribuciones distintas) : {mmd_mnist_fashion:.6f}")
print(f"2. MNIST 1 vs MNIST 2 (Misma distribución)           : {mmd_mnist_mnist:.6f}")
print(f"3. Fashion 1 vs Fashion 2 (Misma distribución)       : {mmd_fashion_fashion:.6f}")

**3. Análisis y Conclusiones**

*   **Comparación de Distribuciones en el RKHS:** Al proyectar las imágenes en el espacio de características, los valores de $MMD^2$ reflejan claramente la discrepancia entre dominios. La comparación base (`MNIST 1 vs MNIST 2` y `Fashion 1 vs Fashion 2`) arroja valores cercanos a cero, confirmando empíricamente que provienen de la misma distribución subyacente. En contraste, el $MMD^2$ entre `MNIST` y `Fashion-MNIST` es significativamente mayor, capturando con éxito el cambio de dominio (*domain shift*) entre los dígitos manuscritos y las prendas de vestir[cite: 8].
*   **Influencia del Ancho de Banda ($\sigma$):** El parámetro $\sigma$ actúa como un factor de escala en la geometría del RKHS. Si $\sigma$ es excesivamente grande, la matriz de Gram tiende a una matriz de unos, provocando que la $MMD^2$ colapse a cero para cualquier par de distribuciones. Si $\sigma$ es demasiado pequeño, la matriz de Gram tiende a la identidad, perdiendo la capacidad de capturar la estructura topológica de los datos. El uso de la heurística de la mediana garantiza que el kernel opere exactamente en la escala intrínseca de las distancias euclidianas de la muestra.
*   **Variabilidad Muestral y Contraste de Hipótesis:** Aunque los valores base entre particiones del mismo dataset son muy pequeños, no son exactamente cero. Esto se debe al **sesgo del estimador** y a la **variabilidad muestral** (muestras finitas de $n=500$). Por esta razón, interpretar un valor aislado de $MMD^2$ como evidencia absoluta de diferencia distribucional tiene limitaciones. Para afirmar estadísticamente que $P_X \neq P_Y$, se requiere un **procedimiento formal de contraste de hipótesis** (por ejemplo, tests de permutación) que permita establecer un umbral de significancia bajo la hipótesis nula y descartar que la discrepancia observada sea producto del azar.

## 3. Índice de correlación de Pearson: espacio original y RKHS

### 3.1 Coeficiente de Pearson en el espacio original

Sean $x, y \in \mathbb{R}^{N \times 1}$ los vectores columna que contienen $N$ observaciones emparejadas de las variables aleatorias escalares $X$ e $Y$. La versión muestral del coeficiente de Pearson estima las esperanzas matemáticas poblacionales utilizando las medias y varianzas empíricas.

**1. Centrado de los datos:**
La media muestral de $x$ se puede escribir matricialmente como $\bar{x} = \frac{1}{N} \mathbf{1}_N^\top x$, donde $\mathbf{1}_N \in \mathbb{R}^{N \times 1}$ es un vector de unos.
El vector centrado $x_c \in \mathbb{R}^{N \times 1}$, al que se le resta la media a cada componente, es:
$$x_c = x - \mathbf{1}_N \bar{x} = x - \mathbf{1}_N \left(\frac{1}{N} \mathbf{1}_N^\top x\right) = \left(I_N - \frac{1}{N} \mathbf{1}_N \mathbf{1}_N^\top\right)x = Hx$$
Donde $H = I_N - \frac{1}{N} \mathbf{1}_N \mathbf{1}_N^\top \in \mathbb{R}^{N \times N}$ es la matriz de centrado.
*Propiedades de $H$:* Es simétrica ($H^\top = H$) e idempotente ($H^2 = H$).

**2. Covarianza y Varianzas Muestrales:**
La suma de productos cruzados centrados (el numerador del coeficiente) es el producto interno euclidiano de los vectores centrados:
$$\sum_{i=1}^N (x_i - \bar{x})(y_i - \bar{y}) = x_c^\top y_c = (Hx)^\top (Hy) = x^\top H^\top H y$$
Usando la simetría y la idempotencia de $H$ ($H^\top H = H^2 = H$), obtenemos:
$$x_c^\top y_c = x^\top H y \quad \text{(Escalar: } (1 \times N) \times (N \times N) \times (N \times 1)\text{)}$$

De manera análoga, las sumas de cuadrados (que definen las varianzas empíricas en el denominador) son:
$$\sum_{i=1}^N (x_i - \bar{x})^2 = x_c^\top x_c = x^\top H x$$
$$\sum_{i=1}^N (y_i - \bar{y})^2 = y_c^\top y_c = y^\top H y$$

**3. Formulación Matricial:**
Sustituyendo estas sumatorias en la definición clásica y cancelando el factor de normalización $\frac{1}{N-1}$ que aparece tanto en el numerador como en la raíz del denominador, llegamos a la expresión solicitada:
$$\hat{\rho}_{XY} = \frac{x^\top H y}{\sqrt{(x^\top H x)(y^\top H y)}}$$

---

### 3.2 Correlación matricial en RKHS

Ahora consideramos variables vectoriales $X \in \mathbb{R}^P$ y $Y \in \mathbb{R}^Q$ proyectadas a un RKHS. Las matrices de Gram $K_X, K_Y \in \mathbb{R}^{N \times N}$ evalúan los kernels sobre las $N$ observaciones.

**1. Centrado en el RKHS:**
Para calcular la covarianza en el espacio de características, las matrices de Gram deben centrarse. El centrado matricial equivale a proyectar las características empíricas al origen en el RKHS. La matriz de Gram centrada es:
$$K_{X,c} = H K_X H \in \mathbb{R}^{N \times N}$$

**2. Índice de Alineamiento (Producto Interno de Frobenius):**
El numerador del índice de alineamiento es el producto interno de Frobenius entre las matrices de Gram centradas: $\langle K_{X,c}, K_{Y,c} \rangle_F = \text{tr}(K_{X,c}^\top K_{Y,c})$.
Dado que las matrices de Gram y $H$ son simétricas, $K_{X,c}$ también lo es. Por lo tanto:
$$\text{tr}(K_{X,c}^\top K_{Y,c}) = \text{tr}((H K_X H) (H K_Y H))$$
Usando la idempotencia $H^2 = H$ y la propiedad cíclica de la traza, el numerador se simplifica a:
$$\text{tr}(K_X H K_Y H)$$

Normalizando este producto interno por la norma de Frobenius de cada matriz (donde $||K_{X,c}||_F = \sqrt{\text{tr}(K_X H K_X H)}$), obtenemos el índice de alineamiento centrado:
$$\rho_{RKHS}(K_X, K_Y) = \frac{\text{tr}(K_X H K_Y H)}{\sqrt{\text{tr}(K_X H K_X H) \text{tr}(K_Y H K_Y H)}}$$

**3. Relación con Pearson para Kernels Lineales ($P=Q=1$):**
Si usamos un kernel lineal $k(u,v) = u^\top v$ sobre variables escalares, las matrices de Gram son matrices de rango 1: $K_X = x x^\top$ y $K_Y = y y^\top$.
La matriz centrada es: $K_{X,c} = H(x x^\top)H = (Hx)(Hx)^\top = x_c x_c^\top$.
Evaluamos el numerador usando la propiedad cíclica de la traza:
$$\text{tr}(K_{X,c} K_{Y,c}) = \text{tr}(x_c x_c^\top y_c y_c^\top) = \text{tr}(x_c^\top y_c y_c^\top x_c)$$
Como $x_c^\top y_c = x^\top H y$ es un escalar, podemos extraerlo:
$$\text{tr}(K_{X,c} K_{Y,c}) = (x^\top H y)^2$$
De forma análoga, para el denominador: $\text{tr}(K_{X,c} K_{X,c}) = (x^\top H x)^2$.
Sustituyendo en la fórmula del alineamiento:
$$\rho_{RKHS} = \frac{(x^\top H y)^2}{\sqrt{(x^\top H x)^2 (y^\top H y)^2}} = \left( \frac{x^\top H y}{\sqrt{(x^\top H x)(y^\top H y)}} \right)^2 = \hat{\rho}_{XY}^2$$

**Pérdida del signo:**
Se demuestra que $\rho_{RKHS} = \hat{\rho}_{XY}^2$. La pérdida del signo ocurre porque el índice de alineamiento evalúa la similitud (producto interno) entre dos matrices semidefinidas positivas en el RKHS. Geométricamente, mide la magnitud del alineamiento de los subespacios inducidos por las características, lo cual es estrictamente no negativo, perdiendo la noción de "dirección" (correlación inversa) que sí posee la variable aleatoria 1D en el espacio original.